In [0]:
data = spark.read.format("text") \
    .option("recursiveFileLookup", "true") \
    .load("/Volumes/workspace/default/bronze/maildir/")

In [0]:
data.printSchema()

root
 |-- value: string (nullable = true)



In [0]:
data.show(20, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                         |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Message-ID: <12090449.1075863678315.

In [0]:
data.count()

766293

In [0]:
from pyspark.sql.functions import col

data = spark.read.format("text") \
    .option("recursiveFileLookup", "true") \
    .load("/Volumes/workspace/default/bronze/maildir/") \
    .withColumn("file_path", col("_metadata.file_path"))

In [0]:
data.show(5, truncate=False)

+---------------------------------------------------------+-----------------------------------------------------------------------+
|value                                                    |file_path                                                              |
+---------------------------------------------------------+-----------------------------------------------------------------------+
|Message-ID: <12090449.1075863678315.JavaMail.evans@thyme>|dbfs:/Volumes/workspace/default/bronze/maildir/presto-k/sent_items/1103|
|Date: Tue, 22 May 2001 20:18:00 -0700 (PDT)              |dbfs:/Volumes/workspace/default/bronze/maildir/presto-k/sent_items/1103|
|From: no.address@enron.com                               |dbfs:/Volumes/workspace/default/bronze/maildir/presto-k/sent_items/1103|
|To: na-incoming@enron.com                                |dbfs:/Volumes/workspace/default/bronze/maildir/presto-k/sent_items/1103|
|Subject: kpresto                                         |dbfs:/Volumes/wor

In [0]:
from pyspark.sql.functions import collect_list, concat_ws

email_data = data.groupBy("file_path") \
    .agg(concat_ws("\n", collect_list("value")).alias("raw_email"))

In [0]:
email_data.count()

13190

In [0]:
email_data.show(3, truncate=False)

+---------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
from pyspark.sql.functions import split

email_split = email_data.withColumn(
    "header",
    split("raw_email", "\n\n").getItem(0)
).withColumn(
    "body",
    split("raw_email", "\n\n").getItem(1)
)

In [0]:
#extract needed columns
from pyspark.sql.functions import regexp_extract

structured_data = email_split.select(
    "file_path",
    regexp_extract("header", r"(?m)^From: (.*)", 1).alias("from_email"),
    regexp_extract("header", r"(?m)^To: (.*)", 1).alias("to_email"),
    regexp_extract("header", r"(?m)^Subject: (.*)", 1).alias("subject"),
    regexp_extract("header", r"(?m)^Date: (.*)", 1).alias("date"),
    "body"
)

In [0]:
clean_data = structured_data.filter(structured_data.from_email != "")

In [0]:
from pyspark.sql.functions import regexp_extract, lower, trim

#filter from_mail
clean_data = clean_data.withColumn(
    "from_email",
    lower(
        trim(
            regexp_extract("from_email", r"([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]+)", 1)
        )
    )
)

In [0]:
#filter to_mail
clean_data = clean_data.withColumn(
    "to_email",
    lower(
        trim(
            regexp_extract("to_email", r"([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]+)", 1)
        )
    )
)

In [0]:
clean_data.show(10, truncate=False)

+---------------------------------------------------------------+-----------------------------------+----------------------------------+-----------------------------------------------------------+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file_path                       

In [0]:
from pyspark.sql.functions import col, trim

clean_data = clean_data.filter(
    col("date").isNotNull() &
    (trim(col("date")) != "")
)

In [0]:
clean_data.show(5, truncate=False)

+---------------------------------------------------------------+-----------------------------------+----------------------------------+------------------------------------------------+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file_path                                  

In [0]:
clean_data.count()

13190

In [0]:
#filter email
from pyspark.sql.functions import col

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

clean_data = clean_data.filter(
    col("from_email").rlike(email_regex) &
    col("to_email").rlike(email_regex)
)

In [0]:
#filter empty rows
from pyspark.sql.functions import col, trim

clean_data = clean_data.filter(
    col("date").isNotNull() &
    (trim(col("date")) != "")
)

In [0]:
clean_data.count()

12490

In [0]:
from pyspark.sql.functions import col

null_date_count = clean_data.filter(col("date").isNull()).count()

In [0]:
null_date_count

0

In [0]:
clean_data.select("date").show(25, truncate=False)

+-------------------------------------+
|date                                 |
+-------------------------------------+
|Sun, 30 Dec 2001 23:42:30 -0800 (PST)|
|Tue, 1 Jan 2002 14:46:05 -0800 (PST) |
|Mon, 29 Oct 2001 15:48:08 -0800 (PST)|
|Mon, 29 Oct 2001 16:15:38 -0800 (PST)|
|Mon, 29 Oct 2001 18:23:36 -0800 (PST)|
|Fri, 23 Nov 2001 09:11:38 -0800 (PST)|
|Tue, 27 Nov 2001 08:02:04 -0800 (PST)|
|Sat, 29 Dec 2001 15:02:59 -0800 (PST)|
|Mon, 26 Nov 2001 08:31:11 -0800 (PST)|
|Wed, 6 Sep 2000 07:02:00 -0700 (PDT) |
|Wed, 6 Sep 2000 07:00:00 -0700 (PDT) |
|Wed, 6 Sep 2000 04:46:00 -0700 (PDT) |
|Thu, 31 Aug 2000 05:02:00 -0700 (PDT)|
|Fri, 25 Aug 2000 06:32:00 -0700 (PDT)|
|Fri, 25 Aug 2000 01:58:00 -0700 (PDT)|
|Fri, 25 Aug 2000 01:47:00 -0700 (PDT)|
|Mon, 21 Aug 2000 09:40:00 -0700 (PDT)|
|Sun, 20 Aug 2000 10:38:00 -0700 (PDT)|
|Tue, 8 Aug 2000 09:31:00 -0700 (PDT) |
|Tue, 21 Nov 2000 03:23:00 -0800 (PST)|
|Mon, 7 Aug 2000 05:43:00 -0700 (PDT) |
|Wed, 26 Jul 2000 03:56:00 -0700 (PDT)|


In [0]:
#disable spark strict parser
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [0]:
from pyspark.sql.functions import regexp_replace, trim
#remove timezone words
clean_data = clean_data.withColumn(
    "normalized_date",
    trim(regexp_replace("date", r"\s*\([A-Za-z]+\)", ""))
)

In [0]:
clean_data.select("date", "normalized_date").show(25, truncate=False)

+-------------------------------------+-------------------------------+
|date                                 |normalized_date                |
+-------------------------------------+-------------------------------+
|Sun, 30 Dec 2001 23:42:30 -0800 (PST)|Sun, 30 Dec 2001 23:42:30 -0800|
|Tue, 1 Jan 2002 14:46:05 -0800 (PST) |Tue, 1 Jan 2002 14:46:05 -0800 |
|Mon, 29 Oct 2001 15:48:08 -0800 (PST)|Mon, 29 Oct 2001 15:48:08 -0800|
|Mon, 29 Oct 2001 16:15:38 -0800 (PST)|Mon, 29 Oct 2001 16:15:38 -0800|
|Mon, 29 Oct 2001 18:23:36 -0800 (PST)|Mon, 29 Oct 2001 18:23:36 -0800|
|Fri, 23 Nov 2001 09:11:38 -0800 (PST)|Fri, 23 Nov 2001 09:11:38 -0800|
|Tue, 27 Nov 2001 08:02:04 -0800 (PST)|Tue, 27 Nov 2001 08:02:04 -0800|
|Sat, 29 Dec 2001 15:02:59 -0800 (PST)|Sat, 29 Dec 2001 15:02:59 -0800|
|Mon, 26 Nov 2001 08:31:11 -0800 (PST)|Mon, 26 Nov 2001 08:31:11 -0800|
|Wed, 6 Sep 2000 07:02:00 -0700 (PDT) |Wed, 6 Sep 2000 07:02:00 -0700 |
|Wed, 6 Sep 2000 07:00:00 -0700 (PDT) |Wed, 6 Sep 2000 07:00:00 

In [0]:
from pyspark.sql.functions import to_timestamp
#convert to timestamp
clean_data = clean_data.withColumn(
    "email_timestamp",
    to_timestamp("normalized_date", "EEE, d MMM yyyy HH:mm:ss Z")
)

In [0]:
clean_data.show(10, truncate=False)

+---------------------------------------------------------------+-----------------------------------+----------------------------------+-----------------------------------------------------------+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------+--

In [0]:
clean_data = clean_data.drop("date", "normalized_date") \
    .withColumnRenamed("email_timestamp", "date")

In [0]:
import pandas as pd
display(pd.DataFrame(clean_data.columns, columns=["Column Name"]))

Column Name
file_path
from_email
to_email
subject
body
date


In [0]:
from pyspark.sql.functions import year, month, dayofmonth, dayofweek, hour, date_format

clean_data = clean_data \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("day_of_week", date_format("date", "E")) \
    .withColumn("hour", hour("date"))

In [0]:
display(pd.DataFrame(clean_data.columns, columns=["Column Name"]))

Column Name
file_path
from_email
to_email
subject
body
date
year
month
day
day_of_week


In [0]:
#display emails per year
emails_per_year = clean_data.groupBy("year").count().orderBy("year")
emails_per_year.display()

year,count
1,8
2,13
1999,10
2000,1699
2001,7864
2002,2851
2004,41
2043,1
2044,3


In [0]:
#filter out bad years
clean_data = clean_data.filter(~clean_data.year.isin([1, 2, 2043, 2044]))

In [0]:
emails_per_year = clean_data.groupBy("year").count().orderBy("year")
emails_per_year.display()

year,count
1999,10
2000,1699
2001,7864
2002,2851
2004,41


In [0]:
emails_per_month = clean_data.groupBy("month") \
                              .count() \
                              .orderBy("month")

emails_per_month.display()

month,count
1,1781
2,1512
3,411
4,252
5,662
6,604
7,415
8,600
9,598
10,2721


In [0]:
emails_by_day = clean_data.groupBy("day_of_week") \
                          .count() \
                          .orderBy("count", ascending=False)

emails_by_day.display()

day_of_week,count
Wed,2639
Tue,2578
Mon,2480
Thu,2253
Fri,2076
Sat,264
Sun,175


In [0]:
emails_by_hour = clean_data.groupBy("hour") \
                           .count() \
                           .orderBy("hour")

emails_by_hour.display()

hour,count
0,337
1,259
2,185
3,132
4,87
5,36
6,71
7,100
8,226
9,388


In [0]:
clean_data.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/email_warehouse/silver/clean_emails"
)